# ETL Silver - Temperatura diaria METAR + INMET

Genera temperatura media, minima y maxima diaria por estacion, unificando METAR (aeropuertos,
cobertura nacional) e INMET (estaciones automaticas dedicadas dentro de la cuenca, Fase 3 del
roadmap). Cada fila lleva `estacion_id` (clave generica) y `fuente` (`metar`|`inmet`) para
trazabilidad. R8 (Decision 019): sin umbral de exclusion, la cobertura real se mide y se
publica en `attribute_quality` a titulo informativo -- el escopeo real a `alta_frontera` (solo
estaciones INMET, ver Decision 025) se aplica en Gold, no aca.

In [ ]:
from datetime import timedelta

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

METAR_BRONZE_TABLE = 'weather.bronze.metar'
INMET_BRONZE_TABLE = 'weather.bronze.inmet'
TARGET_TABLE = 'weather.silver.temperature_daily'
QUALITY_TABLE = 'weather.silver.attribute_quality'
THRESHOLD_PCT = 0.90

try:
    dbutils.widgets.dropdown('load_mode', 'incremental', ['full', 'incremental'])
    dbutils.widgets.text('incremental_lookback_days', '14')
    load_mode = dbutils.widgets.get('load_mode')
    incremental_lookback_days = int(dbutils.widgets.get('incremental_lookback_days'))
except Exception:
    load_mode = 'incremental'
    incremental_lookback_days = 14

print(f'load_mode={load_mode}, incremental_lookback_days={incremental_lookback_days}')

In [ ]:
def ensure_columns(df, columns):
    for column_name in columns:
        if column_name not in df.columns:
            df = df.withColumn(column_name, F.lit(None).cast('string'))
    return df


def apply_incremental_window(df):
    if load_mode == 'full':
        return df

    max_target_fecha = spark.table(TARGET_TABLE).agg(F.max('fecha').alias('max_fecha')).first()['max_fecha']
    if max_target_fecha is None:
        return df

    start_date = max_target_fecha - timedelta(days=incremental_lookback_days)
    print(f'Processing Silver temperature from {start_date}')
    return df.filter(F.col('fecha') >= F.lit(start_date))


def build_quality(df, attribute_name, notes):
    return (
        df.agg(
            F.min('fecha').alias('evaluation_start_date'),
            F.max('fecha').alias('evaluation_end_date'),
            F.countDistinct(F.when(F.col(attribute_name).isNotNull(), F.col('fecha'))).cast('bigint').alias('observed_days'),
        )
        .withColumn('expected_days', F.when(F.col('evaluation_start_date').isNull(), F.lit(0)).otherwise(F.datediff(F.col('evaluation_end_date'), F.col('evaluation_start_date')) + F.lit(1)).cast('bigint'))
        .withColumn('missing_days', F.greatest(F.col('expected_days') - F.col('observed_days'), F.lit(0)).cast('bigint'))
        .withColumn('missing_pct', F.when(F.col('expected_days') == 0, F.lit(1.0)).otherwise(F.col('missing_days') / F.col('expected_days')))
        .withColumn('threshold_pct', F.lit(THRESHOLD_PCT))
        .withColumn('is_usable', F.col('missing_pct') <= F.col('threshold_pct'))
        .withColumn('source_layer', F.lit('silver'))
        .withColumn('source_table', F.lit(TARGET_TABLE))
        .withColumn('source_name', F.lit('temperature_daily'))
        .withColumn('attribute_name', F.lit(attribute_name))
        .withColumn('grain', F.lit('global_source_daily'))
        .withColumn('evaluated_at', F.current_timestamp())
        .withColumn('notes', F.lit(notes))
        .withColumn('created_at', F.current_timestamp())
        .withColumn('updated_at', F.current_timestamp())
        .select('source_layer', 'source_table', 'source_name', 'attribute_name', 'grain', 'evaluation_start_date', 'evaluation_end_date', 'expected_days', 'observed_days', 'missing_days', 'missing_pct', 'threshold_pct', 'is_usable', 'evaluated_at', 'notes', 'created_at', 'updated_at')
    )


def merge_quality(quality_df):
    DeltaTable.forName(spark, QUALITY_TABLE).alias('t').merge(
        quality_df.alias('s'),
        't.source_table = s.source_table AND t.attribute_name = s.attribute_name AND t.grain = s.grain',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


def merge_daily(daily_df):
    if daily_df.limit(1).count() == 0:
        print('No temperature rows to merge')
        return

    DeltaTable.forName(spark, TARGET_TABLE).alias('t').merge(
        daily_df.alias('s'),
        't.fecha = s.fecha AND t.estacion_id = s.estacion_id',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [ ]:
DAILY_COLUMNS = [
    'fecha', 'estacion_id', 'icao_id', 'fuente', 'temp_media_c', 'temp_min_c', 'temp_max_c',
    'registros_total', 'registros_validos', 'first_obs_ts', 'last_obs_ts', 'source_table',
    'processed_at', 'updated_at',
]

# --- METAR (aeropuertos, cobertura nacional) ---
raw_metar = ensure_columns(spark.table(METAR_BRONZE_TABLE), ['icaoId', 'stationId', 'obsTime', 'reportTime', 'temp', 'tmpf'])

bronze_metar = (
    raw_metar.select('icaoId', 'stationId', 'obsTime', 'reportTime', 'temp', 'tmpf')
    .withColumn('icao_id', F.coalesce(F.col('icaoId').cast('string'), F.col('stationId').cast('string')))
    .withColumn('obs_ts', F.coalesce(F.to_timestamp(F.from_unixtime(F.col('obsTime').cast('long'))), F.to_timestamp('reportTime')))
    .withColumn('report_ts', F.to_timestamp('reportTime'))
    .withColumn('obs_key', F.coalesce(F.col('obsTime').cast('string'), F.col('reportTime').cast('string')))
    .withColumn('temp_c', F.coalesce(F.col('temp').cast('double'), (F.col('tmpf').cast('double') - F.lit(32.0)) * F.lit(5.0) / F.lit(9.0)))
    .withColumn('fecha', F.to_date('obs_ts'))
    .filter(F.col('icao_id').isNotNull())
    .filter(F.col('fecha').isNotNull())
    .filter((F.col('temp_c').isNull()) | ((F.col('temp_c') >= F.lit(-80.0)) & (F.col('temp_c') <= F.lit(60.0))))
)

metar_dedupe_window = Window.partitionBy('icao_id', 'obs_key').orderBy(F.col('report_ts').desc_nulls_last())
bronze_metar = bronze_metar.withColumn('rn', F.row_number().over(metar_dedupe_window)).filter(F.col('rn') == 1).drop('rn')
bronze_metar = apply_incremental_window(bronze_metar)

metar_daily = (
    bronze_metar.groupBy('fecha', 'icao_id')
    .agg(
        F.avg('temp_c').alias('temp_media_c'),
        F.min('temp_c').alias('temp_min_c'),
        F.max('temp_c').alias('temp_max_c'),
        F.count('*').cast('bigint').alias('registros_total'),
        F.count('temp_c').cast('bigint').alias('registros_validos'),
        F.min('obs_ts').alias('first_obs_ts'),
        F.max('obs_ts').alias('last_obs_ts'),
    )
    .withColumn('estacion_id', F.col('icao_id'))
    .withColumn('fuente', F.lit('metar'))
    .withColumn('source_table', F.lit(METAR_BRONZE_TABLE))
    .withColumn('processed_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
    .select(*DAILY_COLUMNS)
)

# --- INMET (estaciones automaticas dentro de la cuenca, Fase 3) ---
bronze_inmet = (
    spark.table(INMET_BRONZE_TABLE)
    .withColumn('medicao_ts', F.to_timestamp('data_hora_medicao'))
    .withColumn('fecha', F.to_date('medicao_ts'))
    .filter(F.col('codigo_estacao').isNotNull())
    .filter(F.col('fecha').isNotNull())
    .filter((F.col('temp_c').isNull()) | ((F.col('temp_c') >= F.lit(-80.0)) & (F.col('temp_c') <= F.lit(60.0))))
    .dropDuplicates(['codigo_estacao', 'medicao_ts'])
)
bronze_inmet = apply_incremental_window(bronze_inmet)

inmet_daily = (
    bronze_inmet.groupBy('fecha', 'codigo_estacao')
    .agg(
        F.avg('temp_c').alias('temp_media_c'),
        F.min('temp_c').alias('temp_min_c'),
        F.max('temp_c').alias('temp_max_c'),
        F.count('*').cast('bigint').alias('registros_total'),
        F.count('temp_c').cast('bigint').alias('registros_validos'),
        F.min('medicao_ts').alias('first_obs_ts'),
        F.max('medicao_ts').alias('last_obs_ts'),
    )
    .withColumn('estacion_id', F.col('codigo_estacao'))
    .withColumn('icao_id', F.lit(None).cast('string'))
    .withColumn('fuente', F.lit('inmet'))
    .withColumn('source_table', F.lit(INMET_BRONZE_TABLE))
    .withColumn('processed_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
    .select(*DAILY_COLUMNS)
)

daily = metar_daily.unionByName(inmet_daily)

merge_daily(daily)

full_daily_for_quality = spark.table(TARGET_TABLE)
for attribute_name in ['temp_media_c', 'temp_min_c', 'temp_max_c']:
    merge_quality(build_quality(full_daily_for_quality, attribute_name, f'Temperatura diaria METAR+INMET: {attribute_name}'))

spark.table(TARGET_TABLE).groupBy('fuente').agg(F.min('fecha').alias('inicio'), F.max('fecha').alias('fin'), F.count('*').alias('rows'), F.countDistinct('estacion_id').alias('estaciones')).show()